In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV

# Read Data

In [ ]:
train_dataset = pd.read_csv("intermediate_data/train_dataset.csv")
test_dataset = pd.read_csv("./intermediate_data/test_dataset.csv")

# Light preprocessing

### Drop all NaNs columns
These columns are created in preprocessing when no record exists for a lookback period with a given filtering criteria

In [ ]:
cols_all_nan = train_dataset.columns[train_dataset.isna().all()]
train_dataset = train_dataset.drop(columns=cols_all_nan)
test_dataset = test_dataset.drop(columns=cols_all_nan, errors="ignore")

In [ ]:
X = train_dataset.drop(["SK_ID_CURR", "TARGET", 'Unnamed: 0'], axis=1)
y = train_dataset["TARGET"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
num_cols = X_train.select_dtypes(include="number").columns.to_list()
cat_cols = X_train.columns.difference(num_cols).to_list()
cat_cols

In [ ]:
num_pipeline = Pipeline(
    [
        ("num_impute", SimpleImputer(strategy='mean')),
        ("sc", StandardScaler())
    ]
)
cat_pipeline = Pipeline([
        ("cat_impute", SimpleImputer(strategy='most_frequent', add_indicator=True)),
        ("ohe", OneHotEncoder())
    ]
)
ct = ColumnTransformer(
    [
        ("num_preprocess", num_pipeline, num_cols),
        ("cat_preprocess", cat_pipeline, cat_cols)
    ],
    verbose_feature_names_out=False
)

# Newton Cholesky without l1 or elastic net

In [ ]:
logreg_pipe_nc = Pipeline([
    ("ct", ct),
    ("model", LogisticRegression(
        solver='newton-cholesky',
        C=1,
        class_weight='balanced',
        max_iter=500,
        random_state=42,
        verbose=1
    ))
])

In [ ]:
logreg_pipe_nc.fit(X_train, y_train)

In [ ]:
proba = logreg_pipe_nc.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, proba)

# "saga" with elasticnet and gridsearchcv

In [ ]:
logreg_pipe = Pipeline([
    ("ct", ct),
    ("model", LogisticRegression(
        solver='saga',
        class_weight='balanced',
        max_iter=1000,
        tol=1e-3,
        random_state=42,
    ))
])

In [ ]:
param_dist = {
    "model__l1_ratio": [0, 0.1, 0.5, 0.9, 1],
    "model__C": [1, 0.1, 0.01],
}

In [ ]:
grid_search = GridSearchCV(
    estimator=logreg_pipe,
    param_grid=param_dist,
    scoring='roc_auc',
    n_jobs=6,
    cv=5,
    verbose=1,
)

In [ ]:
grid_search.fit(X_train, y_train)

In [ ]:
best_estimator = grid_search.best_estimator_
coef = best_estimator.named_steps["model"].coef_[0]
feature_importance_df = pd.DataFrame({
    "feature": best_estimator.named_steps['ct'].get_feature_names_out(),
    "coef": coef
})

selected_features = feature_importance_df[feature_importance_df["coef"] != 0]



In [ ]:
selected_features.sort_values("coef", key=abs, ascending=False).head(40)

In [ ]:
proba = best_estimator.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, proba)

In [ ]:
proba = logreg_pipe_nc.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, proba)